# 02 · The e-mail test: what did each e-mail do?

Every choice below was fixed in the analysis plan
([`reports/hillstrom_analysis_plan.md`](../reports/hillstrom_analysis_plan.md)) before this
notebook ran: the metrics, the spend cap, the covariates for regression adjustment, the three
comparisons and their multiple-testing correction, and the decision rule.

- **Primary:** spend per customer over the two weeks (intention to treat), capped at the 99th
  percentile of positive spend.
- **Secondary:** conversion rate and visit rate.
- **Comparisons:** men's vs no e-mail, women's vs no e-mail, men's vs women's; Holm at 5% within
  each metric.
- **Adjustment:** Lin (2013) regression adjustment on the pre-send variables, HC2 errors; the
  decision uses the adjusted estimate, the unadjusted one is shown beside it.

In [1]:
import sys
from pathlib import Path


def _root() -> Path:
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "src" / "data" / "hillstrom.py").exists():
            return p
    raise FileNotFoundError("run from inside the project folder")


ROOT = _root()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src import records
from src.analysis import readout
from src.data import hillstrom

df, SOURCE = hillstrom.load()
COVARIATES = ["history", "recency", "mens", "womens", "newbie", "zip_code", "channel"]   # fixed in the plan
MARGIN, COST = 0.40, 0.10                                                                # assumptions in the plan
COMPARISONS = [("mens", "none"), ("womens", "none"), ("mens", "womens")]
CAP = readout.spend_cap(df["spend"])
df["spend_capped"] = df["spend"].clip(upper=CAP)
print(f"source: {SOURCE}; {len(df):,} customers; spend cap ${CAP:,.2f}")

source: real; 64,000 customers; spend cap $499.00


## 1. The effects

Each e-mail against no e-mail, and the two e-mails against each other, on every metric. Spend is
shown capped (the primary) and uncapped.

In [2]:
METRICS = {"spend_capped": "Spend per customer, capped ($)", "spend": "Spend per customer, uncapped ($)",
           "conversion": "Conversion rate", "visit": "Visit rate"}
rows, effects = [], {}
for m in METRICS:
    adj = [readout.lin_adjusted(df, m, t, c, COVARIATES) for t, c in COMPARISONS]
    raw = [readout.diff_in_means(df, m, t, c) for t, c in COMPARISONS]
    holm = readout.holm(adj)
    effects[m] = {"adjusted": adj, "unadjusted": raw, "holm": holm}
    for a, r, h in zip(adj, raw, holm):
        rows.append({"metric": METRICS[m], "comparison": f"{a.treatment} vs {a.control}",
                     "control mean": a.mean_c, "adjusted diff": a.diff, "95% CI low": a.ci[0],
                     "95% CI high": a.ci[1], "relative": a.rel, "p": a.p_value, "Holm 5%": h,
                     "unadjusted diff": r.diff, "CI narrower by": 1 - a.se / r.se})
table = pd.DataFrame(rows)
table.round(4)

,metric,comparison,control mean,adjusted diff,95% CI low,95% CI high,relative,p,Holm 5%,unadjusted diff,CI narrower by
0,"Spend per customer, capped ($)",mens vs none,0.6528,0.7669,0.4826,1.0512,1.1748,0.0000,True,0.7698,0.0012
1,"Spend per customer, capped ($)",womens vs none,0.6528,0.4247,0.1694,0.6799,0.6505,0.0011,True,0.4244,0.0008
2,"Spend per customer, capped ($)",mens vs womens,1.0772,0.3397,0.0274,0.6520,0.3153,0.0330,True,0.3454,0.0017
3,"Spend per customer, uncapped ($)",mens vs none,0.6528,0.7669,0.4826,1.0512,1.1748,0.0000,True,0.7698,0.0012
4,"Spend per customer, uncapped ($)",womens vs none,0.6528,0.4247,0.1694,0.6799,0.6505,0.0011,True,0.4244,0.0008
5,"Spend per customer, uncapped ($)",mens vs womens,1.0772,0.3397,0.0274,0.6520,0.3153,0.0330,True,0.3454,0.0017
6,Conversion rate,mens vs none,0.0057,0.0068,0.0050,0.0086,1.1829,0.0000,True,0.0068,0.0014
7,Conversion rate,womens vs none,0.0057,0.0031,0.0015,0.0047,0.5436,0.0002,True,0.0031,0.0011
8,Conversion rate,mens vs womens,0.0088,0.0036,0.0017,0.0056,0.4119,0.0002,True,0.0037,0.0012
9,Visit rate,mens vs none,0.1062,0.0761,0.0695,0.0826,0.7164,0.0000,True,0.0766,0.0148


## 2. What regression adjustment bought

The pre-send variables explain part of each customer's outcome, so removing that part narrows
the interval without moving the estimate (the arms were randomized, so the adjustment should
change the point estimate by little). The width of the interval is reported, and the
equivalent extra sample: an interval that is narrower by a share `s` would otherwise need
1/(1-s)^2 times as many customers.

In [3]:
gain = table.groupby("metric", sort=False)["CI narrower by"].mean()
gain_sample = 1 / (1 - gain) ** 2 - 1
pd.DataFrame({"CI narrower by": gain, "equivalent extra sample": gain_sample}).round(3)

,CI narrower by,equivalent extra sample
metric,,
"Spend per customer, capped ($)",0.001,0.003
"Spend per customer, uncapped ($)",0.001,0.003
Conversion rate,0.001,0.002
Visit rate,0.015,0.030


## 3. The decision rule, applied

From the plan:

1. An e-mail is worth sending to everyone only if its adjusted, capped spend lift over no
   e-mail passes Holm and the lift times the margin (40%) exceeds the cost of sending ($0.10 an
   e-mail).
2. If both pass, the higher one is chosen only if men's vs women's also passes Holm; otherwise
   the memo says they cannot be told apart.
3. Targeting by customer is decided in notebook 03.

In [4]:
prim = {f"{e.treatment} vs {e.control}": (e, h) for e, h in zip(effects["spend_capped"]["adjusted"],
                                                                  effects["spend_capped"]["holm"])}
value = {}
for arm in ("mens", "womens"):
    e, h = prim[f"{arm} vs none"]
    value[arm] = {"lift": e.diff, "ci": list(e.ci), "holm_pass": bool(h),
                  "value_per_1000": readout.value_per_1000(e.diff, MARGIN, COST),
                  "value_per_1000_ci": [readout.value_per_1000(x, MARGIN, COST) for x in e.ci],
                  "breakeven_margin": readout.breakeven_margin(e.diff, COST),
                  "passes": bool(h and e.diff * MARGIN > COST)}
head, head_pass = prim["mens vs womens"]
passing = [a for a in ("mens", "womens") if value[a]["passes"]]
if not passing:
    decision = "none"
elif len(passing) == 1:
    decision = passing[0]
else:
    better = "mens" if head.diff > 0 else "womens"
    decision = better if head_pass else "either"
print({k: {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in v.items()} for k, v in value.items()})
print(f"men's vs women's: {head.diff:+.3f} [{head.ci[0]:+.3f}, {head.ci[1]:+.3f}], Holm pass {bool(head_pass)}")
print("decision under the plan's rule:", decision)

{'mens': {'lift': 0.7669, 'ci': [0.48256587970595194, 1.0512169077994324], 'holm_pass': True, 'value_per_1000': 206.7566, 'value_per_1000_ci': [93.02635188238078, 320.486763119773], 'breakeven_margin': 0.1304, 'passes': True}, 'womens': {'lift': 0.4247, 'ci': [0.16942551914053805, 0.6798928172632516], 'holm_pass': True, 'value_per_1000': 69.8637, 'value_per_1000_ci': [-32.22979234378479, 171.95712690530067], 'breakeven_margin': 0.2355, 'passes': True}}
men's vs women's: +0.340 [+0.027, +0.652], Holm pass True
decision under the plan's rule: mens


## 4. Figure

In [5]:
LABEL = {"mens vs none": "Men's vs no e-mail", "womens vs none": "Women's vs no e-mail",
         "mens vs womens": "Men's vs women's"}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, m, unit in zip(axes, ["spend_capped", "conversion", "visit"], ["$ per customer", "points", "points"]):
    scale = 1 if m.startswith("spend") else 100
    for i, (a, r) in enumerate(zip(effects[m]["adjusted"], effects[m]["unadjusted"])):
        key = f"{a.treatment} vs {a.control}"
        ax.plot([a.ci[0] * scale, a.ci[1] * scale], [i, i], color="#5B8FF9", lw=3)
        ax.plot(a.diff * scale, i, "o", color="#5B8FF9", ms=7, mec="white", mew=1.5)
        ax.plot([r.ci[0] * scale, r.ci[1] * scale], [i + 0.22, i + 0.22], color="#BFBFBF", lw=2)
    ax.axvline(0, color="#2B2B2B", lw=0.8)
    ax.set_yticks(range(3), [LABEL[f"{t} vs {c}"] for t, c in COMPARISONS])
    ax.invert_yaxis()
    ax.set(title=METRICS[m].split(",")[0], xlabel=f"difference ({unit})")
    ax.grid(alpha=0.25, axis="x")
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("The e-mail test: effect of each e-mail over two weeks", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
fig.text(0.99, 0.97, "blue: regression-adjusted 95% CI · grey: unadjusted", ha="right", va="top",
         fontsize=9, color="#555555")
records.save_figure(fig, "02_email_test_effects.png", SOURCE)

In [6]:
record = {
    "cap": CAP, "covariates": COVARIATES, "margin_assumed": MARGIN, "cost_per_email_assumed": COST,
    "effects": {m: [{**a.as_dict(), "holm_pass": bool(h), "unadjusted_diff": r.diff,
                     "unadjusted_ci": list(r.ci), "ci_narrower_by": 1 - a.se / r.se}
                    for a, r, h in zip(v["adjusted"], v["unadjusted"], v["holm"])] for m, v in effects.items()},
    "value": value, "head_to_head": {"diff": head.diff, "ci": list(head.ci), "holm_pass": bool(head_pass)},
    "decision": decision,
}
print("wrote", records.write("02_email_test_readout", record, SOURCE).name)

wrote 02_email_test_readout.json


## Verdict

Rendered from the record written above.

In [7]:
DECISION_TEXT = {"none": "send nothing", "mens": "send the men's e-mail to everyone",
                 "womens": "send the women's e-mail to everyone",
                 "either": "send either e-mail; they cannot be told apart"}


def fmt(v):
    return f"{v['lift']:+.2f} per customer [{v['ci'][0]:+.2f}, {v['ci'][1]:+.2f}]"


lines = [
    f"Men's e-mail: spend {fmt(value['mens'])}; worth ${value['mens']['value_per_1000']:,.0f} per 1,000 "
    f"customers e-mailed at a 40% margin (break-even margin {value['mens']['breakeven_margin']:.0%}).",
    f"Women's e-mail: spend {fmt(value['womens'])}; ${value['womens']['value_per_1000']:,.0f} per 1,000 "
    f"(break-even margin {value['womens']['breakeven_margin']:.0%}).",
    f"Men's against women's: {head.diff:+.2f} [{head.ci[0]:+.2f}, {head.ci[1]:+.2f}]; "
    + ("the difference passes Holm." if head_pass else "the two cannot be told apart."),
    f"Regression adjustment narrowed the spend intervals by {gain[METRICS['spend_capped']]:.0%} on average, "
    f"worth {gain_sample[METRICS['spend_capped']]:.0%} more customers.",
    f"Decision under the plan's rule: {DECISION_TEXT[decision]}.",
]
print("\n".join(f"{i}. {t}" for i, t in enumerate(lines, 1)))

1. Men's e-mail: spend +0.77 per customer [+0.48, +1.05]; worth $207 per 1,000 customers e-mailed at a 40% margin (break-even margin 13%).
2. Women's e-mail: spend +0.42 per customer [+0.17, +0.68]; $70 per 1,000 (break-even margin 24%).
3. Men's against women's: +0.34 [+0.03, +0.65]; the difference passes Holm.
4. Regression adjustment narrowed the spend intervals by 0% on average, worth 0% more customers.
5. Decision under the plan's rule: send the men's e-mail to everyone.


## 5. Rollout and follow-up

The decision is to e-mail everyone, but the e-mail's value has to keep being measured: once
everyone gets it, there is no comparison group left. The usual answer is a **permanent
holdback**: a small random share of customers who never get the campaign e-mail.

How small can it be? With 64,000 customers per send, the table shows the smallest spend lift
each holdback size detects in one send, and how many sends must be pooled before it can detect
**half** the lift measured here (an early warning that the e-mail is wearing out).

In [8]:
LIST = len(df)
ctrl_sd = float(df.loc[df["arm"] == "none", "spend_capped"].std())
z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
ARM = decision if decision in ("mens", "womens") else "mens"   # the e-mail being rolled out
target = value[ARM]["lift"] / 2
holdback = []
for share in (0.02, 0.05, 0.10, 0.20):
    n_c, n_t = LIST * share, LIST * (1 - share)
    per_send = z * ctrl_sd * np.sqrt(1 / n_c + 1 / n_t)
    holdback.append({"holdback_share": share, "customers_held_back": int(n_c), "mde_one_send": per_send,
                     "sends_to_detect_half_the_lift": int(np.ceil((per_send / target) ** 2)),
                     "margin_given_up_per_send": share * LIST * value[ARM]["lift"] * MARGIN})
holdback = pd.DataFrame(holdback)
pick = holdback[holdback["holdback_share"] == 0.05].iloc[0]
print(f"a 5% holdback ({int(pick['customers_held_back']):,} customers) gives up about "
      f"${pick['margin_given_up_per_send']:,.0f} of margin a send and detects half the measured lift "
      f"after {int(pick['sends_to_detect_half_the_lift'])} sends pooled")
holdback.round(3)

a 5% holdback (3,200 customers) gives up about $982 of margin a send and detects half the measured lift after 3 sends pooled


,holdback_share,customers_held_back,mde_one_send,sends_to_detect_half_the_lift,margin_given_up_per_send
0,0.02,1280,0.917,6,392.648
1,0.05,3200,0.589,3,981.621
2,0.10,6400,0.428,2,1963.242
3,0.20,12800,0.321,1,3926.484


**Guardrails to watch during rollout.** This data has none, but a real rollout would track
unsubscribes and spam complaints per send (an e-mail that sells this week but loses the
address costs every later send), deliverability, and whether e-mailing men's merchandise to
women's-line buyers shifts their spending rather than adding to it.

**The next test.** Men's against women's was the closest call. The natural next test is the
men's e-mail against a new version, for example one that features the customer's own line.
Sized with notebook 01's calculation, to detect a difference worth the break-even amount:

In [9]:
mens_sd = float(df.loc[df["arm"] == "mens", "spend_capped"].std())
next_test = {"effect_to_detect": COST / MARGIN, "n_per_arm": readout.n_per_arm(mens_sd, COST / MARGIN),
             "n_per_arm_for_head_to_head_seen_here": readout.n_per_arm(mens_sd, abs(head.diff))}
next_test["sends_needed"] = 2 * next_test["n_per_arm"] / LIST
print(f"two arms of {next_test['n_per_arm']:,} customers ({next_test['sends_needed']:.1f} sends of this list) "
      f"to detect a ${next_test['effect_to_detect']:.2f} difference in spend. That is more than notebook 01's "
      f"{readout.n_per_arm(ctrl_sd, COST / MARGIN):,}, because spend varies more among e-mailed customers "
      f"(sd ${mens_sd:.2f} against ${ctrl_sd:.2f}).")
rollout = {"holdback": holdback.to_dict(orient="records"), "next_test": next_test}
record["rollout"] = rollout
print("updated", records.write("02_email_test_readout", record, SOURCE).name)

two arms of 79,170 customers (2.5 sends of this list) to detect a $0.25 difference in spend. That is more than notebook 01's 33,728, because spend varies more among e-mailed customers (sd $17.75 against $11.59).
updated 02_email_test_readout.json
